# Pay for Content — Browser 사용 사례(Amazon Bedrock AgentCore Runtime)

## 개요

이 사용 사례에서는 **Amazon Bedrock AgentCore payments**와 **Amazon Bedrock AgentCore Browser Tool**을 사용하여
사람이 이용하는 paywall 뒤의 content에 자동으로 결제하고 가져오는 Strands agent를 구축한 후
**Amazon Bedrock AgentCore Runtime에 배포**합니다. Agent는 managed cloud browser를 시작하고
paywall page로 이동하여 DOM에 embedded된 x402 payment requirement를 읽고
AgentCore payments를 통해 cryptographic proof를 생성한 다음 paywall UI와 상호 작용하여
잠금 해제된 content를 반환합니다. 이 모든 과정에서 payment 단계에는 사람의 개입이
필요하지 않습니다.

### 사용 사례 세부 정보

| 항목               | 세부 정보                                                     |
|:-------------------|:--------------------------------------------------------------|
| 사용 사례 유형     | 자동 micropayment를 사용하는 agentic browser automation      |
| Agent 유형         | Single                                                        |
| Hosting            | AgentCore Runtime(managed microVM, role 분리)                 |
| Payment protocol   | x402(HTTP 402 Payment Required)                               |
| Agentic Framework  | Strands Agents                                                |
| LLM model          | Anthropic Claude Sonnet 4.6                                   |
| 난이도             | 중급                                                          |
| 사용 SDK           | boto3 + AgentCore SDK + AgentCorePaymentsPlugin(Strands)      |
| Wallet 유형        | Embedded crypto wallet(AgentCore에서 provision, Coinbase CDP) |
| Network            | Base Sepolia testnet(`eip155:84532`), Solana Devnet 지원      |

### 아키텍처

Agent는 AgentCore Runtime에서 `ProcessPaymentRole`로 실행됩니다. Notebook(app backend)은
`ManagementRole`로 로컬에서 실행되고 budget이 있는 payment session을 생성한 다음
session/instrument context와 함께 `InvokeAgentRuntime`을 호출합니다. 배포된 agent는
`AgentCoreBrowser`(managed cloud Chromium session)를 사용하여 paywall page로 이동하고
embedded `<script>` element에서 x402 requirement를 읽은 후 `ProcessPayment`를 호출하여
Base Sepolia에서 USDC proof를 생성하고 page의 JavaScript payment handler를 통해 proof를
제출합니다. x402 facilitator가 on-chain transaction을 검증하면 content의 잠금이
해제되고 agent가 caller에 반환합니다.

```
Notebook (ManagementRole)              AgentCore Runtime (ProcessPaymentRole)
  │                                     ┌─────────────────────────────────────┐
  │ create_payment_session(budget=$1)   │  agent/payment_agent.py             │
  │                                     │  BedrockAgentCoreApp                │
  │ InvokeAgentRuntime(                 │   ├─ AgentCoreBrowser ──→ Browser   │
  │   manager_arn, session_id,    ────► │   ├─ process_x402_payment           │
  │   instrument_id, paywall_url)       │   └─ AgentCorePaymentsPlugin        │
  │                                     │                                     │
  │◄── unlocked content ──────────────  │  Plugin/PaymentManager call:        │
  │                                     │     ProcessPayment                  │
  │ get_payment_session(check spend)    │  Cannot: CreateSession              │
  │                                     │  Cannot: Override budget            │
  │                                     └─────────────────────────────────────┘
```

<div style="text-align:left">
    <img src="images/architecture_browser_paywall.png" width="75%"/>
</div>

### 사용 사례 핵심 기능

* 실제 on-chain micropayment를 사용하는 end-to-end agentic browser automation
* **AgentCore Runtime에서 hosting** — managed microVM, ECR에 배포된 container, 자동 scaling
* **Infrastructure level에서 role 분리 적용** — Runtime container가 `ProcessPaymentRole`을 직접 사용
* AgentCore Browser Tool이 managed cloud Chromium session을 제공하므로 local browser 불필요
* Agent는 private key를 보유하지 않으며 payment signing은 AgentCore payments에 위임
* App backend에서 설정한 payment session의 `maxSpendAmount`를 통해 사람이 payment limit 제어
* **Built-in observability** — CloudWatch GenAI Observability의 Runtime trace 및 metric
* `PAYMENT-SIGNATURE` header와 Base Sepolia testnet의 USDC를 사용하는 x402 v2 protocol
* Provider 독립적인 설계 — agent logic 변경 없이 wallet provider config 교체

## 사전 요구 사항

이 사용 사례를 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* Node.js 20+(AgentCore CLI용)
* AWS credentials 구성 완료(`aws configure`)
* Amazon Bedrock AgentCore access
* Coinbase CDP account — CDP dashboard의 `CDP_API_KEY_NAME`, `CDP_API_KEY_PRIVATE_KEY`, `CDP_WALLET_SECRET`
  (이 예제에서는 Coinbase CDP를 사용하며 StripePrivy도 지원됨 — 3a단계에서 credential provider config 교체)
  > **필수:** Agent를 실행하기 전에 CDP project에서 **Delegated Signing**을 활성화하세요.
  > [portal.cdp.coinbase.com](https://portal.cdp.coinbase.com) → project → **Wallet** → **Embedded Wallets** → **Policies**로 이동하여 **Delegated signing**을 활성화합니다.
  > 이 설정이 없으면 `ProcessPayment`가 delegated signing error와 함께 실패합니다.
* IAM role 생성 완료 — 아직 실행하지 않았다면 `bash setup_roles.sh`를 한 번 실행합니다. 이 사용 사례의
  `setup_roles.sh`는 `ProcessPaymentRole`이 AgentCore Runtime execution role 역할도 하도록
  구성합니다(ECR, CloudWatch, X-Ray, Bedrock model invocation, browser tool).
* AgentCore CLI installed: `npm install -g @aws/agentcore`
* [AWS CDK](https://docs.aws.amazon.com/cdk/v2/guide/getting_started.html) installed (used by the CLI)

  > **Content provider:** Agent를 실행하기 전에 CDK stack을 배포합니다. `cd content-provider && PAY_TO=0x<your-wallet> bash deploy.sh`. `.env`의 `CONTENT_DISTRIBUTION_URL`을 출력된 CloudFront URL로 설정합니다.
  > `AgentCoreBrowser`는 cloud-managed browser이므로 `localhost`에 연결할 수 없습니다.

In [ ]:
!pip install -r requirements.txt --quiet

## 1단계 — 환경 구성

필요한 모든 값을 `.env` 파일에서 불러옵니다. `.env.sample`을 `.env`로 복사하고 값을
입력합니다. 처음 실행할 때 3단계에서 MANAGER_ARN, PAYMENT_CONNECTOR_ID, PAYMENT_INSTRUMENT_ID를 채웁니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

REGION = os.environ.get("AWS_REGION", "us-west-2")

# ── 엔드포인트 ───────────────────────────────────────────────────────────────
# CP: credential provider, manager + connector 설정. REGION에서 구성하며 다른 host를 가리키려면 CP_ENDPOINT override
CP_ENDPOINT = os.environ.get("CP_ENDPOINT", f"https://bedrock-agentcore-control.{REGION}.amazonaws.com")
# DP: instrument, session, payment 처리
DP_ENDPOINT = os.environ.get("DP_ENDPOINT", f"https://bedrock-agentcore.{REGION}.amazonaws.com")

# ── Coinbase CDP credential 설정 ─────────────────────────────────────────────
CDP_API_KEY_NAME = os.environ["CDP_API_KEY_NAME"]
CDP_API_KEY_PRIVATE_KEY = os.environ["CDP_API_KEY_PRIVATE_KEY"]
CDP_WALLET_SECRET = os.environ["CDP_WALLET_SECRET"]

# ── embedded wallet identity 설정 ─────────────────────────────────────────────
WALLET_EMAIL = os.environ.get("WALLET_EMAIL", "")  # embedded wallet용 email linkedAccount

# ── IAM role 설정 ────────────────────────────────────────────────────────────
MANAGEMENT_ROLE_ARN = os.environ["MANAGEMENT_ROLE_ARN"]
PROCESS_PAYMENT_ROLE_ARN = os.environ["PROCESS_PAYMENT_ROLE_ARN"]
CONTROL_PLANE_ROLE_ARN = os.environ["CONTROL_PLANE_ROLE_ARN"]
RESOURCE_RETRIEVAL_ROLE_ARN = os.environ["RESOURCE_RETRIEVAL_ROLE_ARN"]

# ── Provision된 resource ID(3단계에서 채움 — 재실행 시 provision 생략) ─────────
MANAGER_ARN = os.environ.get("MANAGER_ARN", "")
PAYMENT_CONNECTOR_ID = os.environ.get("PAYMENT_CONNECTOR_ID", "")
PAYMENT_INSTRUMENT_ID = os.environ.get("PAYMENT_INSTRUMENT_ID", "")
SESSION_ID = os.environ.get("SESSION_ID", "")

# ── Session 설정 ─────────────────────────────────────────────────────────────
USER_ID = os.environ.get("USER_ID", "test-user-12345")
SESSION_MAX_SPEND = os.environ.get("SESSION_MAX_SPEND", "1.00")
SESSION_EXPIRY_MINUTES = int(os.environ.get("SESSION_EXPIRY_MINUTES", "60"))

# ── network / blockchain 설정 ────────────────────────────────────────────────
# base-sepolia:  eip155:84532                              (기본값, e2e 테스트 완료)
# solana-devnet: solana:EtWTRABZaYq6iMfeYKouRu166VU2xqa1  (placeholder, 아직 테스트하지 않음)
NETWORK_ALIAS = os.environ.get("NETWORK", "base-sepolia")
NETWORK_MAP = {
    "base-sepolia": {
        "caip2": "eip155:84532",
        "botocore_net": "ETHEREUM",
        "usdc_address": "0x036CbD53842c5426634e7929541eC2318f3dCF7e",
    },
    "solana-devnet": {
        "caip2": "solana:EtWTRABZaYq6iMfeYKouRu166VU2xqa1",
        "botocore_net": "SOLANA",
        "usdc_address": "4zMMC9srt5Ri5X14GAgXhaHii3GnPAEERYPJgZJDncDU",  # pragma: allowlist secret
    },
    "base-mainnet": {
        "caip2": "eip155:8453",
        "botocore_net": "ETHEREUM",
        "usdc_address": "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913",
    },
}
if NETWORK_ALIAS not in NETWORK_MAP:
    raise ValueError(f"Unknown NETWORK '{NETWORK_ALIAS}'. Valid: {list(NETWORK_MAP)}")
ACTIVE_NETWORK = NETWORK_MAP[NETWORK_ALIAS]

# ── 콘텐츠 provider 설정 ─────────────────────────────────────────────────────
CONTENT_DISTRIBUTION_URL = os.environ.get(
    "CONTENT_DISTRIBUTION_URL",
    "",
)
PAYWALL_DEMO_URL = f"{CONTENT_DISTRIBUTION_URL}/article/paywall-demo"

print(f"✅ Region:        {REGION}")
print(f"✅ CP:            {CP_ENDPOINT}")
print(f"✅ DP:            {DP_ENDPOINT}")
print(f"✅ Network:       {NETWORK_ALIAS} ({ACTIVE_NETWORK['caip2']})")
print(f"✅ Content URL:   {CONTENT_DISTRIBUTION_URL}")
print(f"✅ Payment limit: ${SESSION_MAX_SPEND} USD")
if MANAGER_ARN:
    print(f"✅ Manager ARN:   {MANAGER_ARN} (loaded from .env — Step 3 will be skipped)")
if SESSION_ID:
    print(f"✅ Session ID:    {SESSION_ID} (loaded from .env — Step 4 will be skipped)")

## 2단계 — AWS Client 초기화

Notebook(app backend)에는 두 AWS client가 필요합니다. Agent는 AgentCore Runtime에서
실행되고 execution role의 ambient credentials를 직접 사용하므로 여기서는 `ProcessPaymentRole`
client를 생성하지 않습니다.

| Client | Endpoint | Service name | 용도 |
|:-------|:---------|:-------------|:---------|
| `cp_client` | CP (`bedrock-agentcore-control`) | `bedrock-agentcore-control` | `CreatePaymentCredentialProvider`, `CreatePaymentManager`, `CreatePaymentConnector` |
| `mgmt_client` | DP (`bedrock-agentcore`) | `bedrock-agentcore` | `CreatePaymentInstrument`, `CreatePaymentSession`, `GetPaymentSession`, `InvokeAgentRuntime` |

**IAM role 분리:**
* `ControlPlaneRole` — credential provider, payment manager, payment connector(control plane)
* `ManagementRole` — payment session 생성 및 관리, 배포된 agent 호출. `ProcessPayment`에 명시적 Deny 적용
* `ProcessPaymentRole` — AgentCore Runtime execution role. Container 시작 시 이 role을 사용합니다. `ProcessPayment` 호출 및 browser tool 사용은 가능하지만 session/instrument 관리에는 명시적 Deny가 적용됩니다.

Notebook 자체에는 `ProcessPaymentRole`이 필요하지 않으며 Runtime에서만 사용합니다. Wallet
balance 확인(3e단계)에서만 검증을 위해 로컬에서 잠시 이 role을 사용합니다.

In [ ]:
import os
from boto3.session import Session
from datetime import datetime

base_session = Session(region_name=REGION)
sts = base_session.client("sts")
ACCOUNT_ID = sts.get_caller_identity()["Account"]
print(f"✅ AWS account: {ACCOUNT_ID}")


def assume_role(role_arn: str, session_name: str) -> Session:
    creds = sts.assume_role(RoleArn=role_arn, RoleSessionName=session_name)["Credentials"]
    sess = Session(
        aws_access_key_id=creds["AccessKeyId"],
        aws_secret_access_key=creds["SecretAccessKey"],
        aws_session_token=creds["SessionToken"],
        region_name=REGION,
    )
    assumed_arn = sess.client("sts").get_caller_identity()["Arn"]
    print(f"  → {assumed_arn}")
    return sess


# ── control plane client 설정(ControlPlaneRole) ──────────────────────────────
print("\nAssuming ControlPlaneRole...")
cp_session = assume_role(CONTROL_PLANE_ROLE_ARN, f"cp-setup-{int(datetime.now().timestamp())}")
cp_client = cp_session.client("bedrock-agentcore-control", endpoint_url=CP_ENDPOINT)
print("✅ CP client ready (CreatePaymentCredentialProvider / CreatePaymentManager / CreatePaymentConnector)")

# ── Management client(ManagementRole) — session 및 InvokeAgentRuntime ────────
print("\nAssuming ManagementRole...")
mgmt_session = assume_role(MANAGEMENT_ROLE_ARN, f"payments-mgmt-{int(datetime.now().timestamp())}")
mgmt_client = mgmt_session.client("bedrock-agentcore", endpoint_url=DP_ENDPOINT)
print("✅ Management client ready (CreatePaymentSession / GetPaymentSession / InvokeAgentRuntime)")

## 3단계 — Embedded Wallet Resource Provision

이 셀은 사용자별로 한 번 실행하여 다음 AgentCore payments resource stack을 생성합니다.
**CredentialProvider → PaymentManager → PaymentConnection → EmbeddedCryptoWallet Instrument**

`MANAGER_ARN`, `PAYMENT_CONNECTOR_ID`, `PAYMENT_INSTRUMENT_ID`가 이미 있다면
`.env`에 설정하고 4단계로 이동합니다.

> **Embedded wallet:** AgentCore가 on-chain wallet을 provision하므로 기존 CDP wallet은
> 필요하지 않습니다. `linkedAccounts` email field가 wallet을 사용자 identity에 연결합니다.
> Coinbase embedded wallet은 동기식으로 provision됩니다(OTP/email verification 단계 없음).
> OTP verification은 StripePrivy embedded wallet에만 필요합니다.

In [ ]:
import uuid
import time

# ── 3a. Credential Provider 생성 ──────────────────────────────────────────────
# Credential provider가 Coinbase CDP API key를 안전하게 저장
# StripePrivy의 경우 credentialProviderVendor="StripePrivy"로 설정하고
# coinbaseCdpConfiguration을 stripePlatformConfiguration으로 교체
#
# 중요(Coinbase CDP): CDP project에서 Delegated Signing을 활성화해야
# ProcessPayment가 성공함. portal.cdp.coinbase.com → project → Wallet → Embedded Wallets → Policies에서 Delegated signing 활성화
cred_resp = cp_client.create_payment_credential_provider(
    name=f"CoinbaseCdp{int(time.time())}",
    credentialProviderVendor="CoinbaseCDP",  # 또는 "StripePrivy"
    providerConfigurationInput={
        "coinbaseCdpConfiguration": {
            "apiKeyId": CDP_API_KEY_NAME,
            "apiKeySecret": CDP_API_KEY_PRIVATE_KEY,
            "walletSecret": CDP_WALLET_SECRET,
        }
    },
)
CREDENTIAL_PROVIDER_ARN = cred_resp["credentialProviderArn"]
print(f"✅ Credential Provider: {CREDENTIAL_PROVIDER_ARN}")

In [ ]:
# ── 3b. Payment Manager 생성 ──────────────────────────────────────────────────
mgr_resp = cp_client.create_payment_manager(
    name=f"PayMgr{int(time.time())}",
    description="AgentCore payments - Pay for Content Browser use case",
    authorizerType="AWS_IAM",
    roleArn=RESOURCE_RETRIEVAL_ROLE_ARN,
    clientToken=str(uuid.uuid4()),
)
MANAGER_ARN = mgr_resp["paymentManagerArn"]
MANAGER_ID = mgr_resp["paymentManagerId"]
print(f"✅ Payment Manager ARN: {MANAGER_ARN}")
print(f"   Manager ID:          {MANAGER_ID}")

In [ ]:
# ── 3c. Payment Connector 생성 ────────────────────────────────────────────────
# Manager를 credential provider에 연결
# StripePrivy의 경우 type="StripePrivy"로 설정하고 credentialProviderConfigurations를 그에 맞게 업데이트
conn_resp = cp_client.create_payment_connector(
    paymentManagerId=MANAGER_ID,
    name=f"CoinbaseConn{int(time.time())}",
    description="Coinbase CDP connector for embedded wallet",
    type="CoinbaseCDP",  # 또는 "StripePrivy"
    credentialProviderConfigurations=[{"coinbaseCDP": {"credentialProviderArn": CREDENTIAL_PROVIDER_ARN}}],
    clientToken=str(uuid.uuid4()),
)
PAYMENT_CONNECTOR_ID = conn_resp["paymentConnectorId"]
print(f"✅ Payment Connector ID: {PAYMENT_CONNECTOR_ID}")

In [ ]:
# ── 3d. Embedded Crypto Wallet Instrument 생성 ────────────────────────────────
# EMBEDDED_CRYPTO_WALLET: AgentCore가 wallet을 provision하므로 기존 CDP
# wallet은 불필요. linkedAccounts email이 wallet을 사용자 identity에 연결
linked_accounts = []
if WALLET_EMAIL:
    linked_accounts = [{"email": {"emailAddress": WALLET_EMAIL}}]

inst_resp = mgmt_client.create_payment_instrument(
    paymentManagerArn=MANAGER_ARN,
    paymentConnectorId=PAYMENT_CONNECTOR_ID,
    userId=USER_ID,
    paymentInstrumentType="EMBEDDED_CRYPTO_WALLET",
    paymentInstrumentDetails={
        "embeddedCryptoWallet": {
            "network": ACTIVE_NETWORK["botocore_net"],  # "ETHEREUM" 또는 "SOLANA"
            "linkedAccounts": linked_accounts,
        }
    },
    clientToken=str(uuid.uuid4()),
)
instrument = inst_resp["paymentInstrument"]
PAYMENT_INSTRUMENT_ID = instrument["paymentInstrumentId"]
wallet_details = instrument.get("paymentInstrumentDetails", {}).get("embeddedCryptoWallet", {})
wallet_address = wallet_details.get("walletAddress", "<pending>")
# WalletHub URL — 사용자가 열어 wallet에 입금하고 signing 권한 부여
WALLET_HUB_URL = wallet_details.get("redirectUrl", "")

print(f"✅ Payment Instrument ID: {PAYMENT_INSTRUMENT_ID}")
print(f"   Wallet Address:        {wallet_address}")
print(f"   Network:               {ACTIVE_NETWORK['caip2']}")
if WALLET_HUB_URL:
    print(f"   WalletHub URL:         {WALLET_HUB_URL}")
print()
print("📋 Save these values to .env for future runs:")
print(f"   MANAGER_ARN={MANAGER_ARN}")
print(f"   PAYMENT_CONNECTOR_ID={PAYMENT_CONNECTOR_ID}")
print(f"   PAYMENT_INSTRUMENT_ID={PAYMENT_INSTRUMENT_ID}")

### WalletHub — Wallet 입금 및 Signing 권한 부여

AgentCore에서 embedded wallet을 provision했습니다. Agent가 payment를 수행하기 전에
**WalletHub**에서 설정을 완료해야 합니다.

1. 위에 출력된 **WalletHub URL을 엽니다.** `WALLET_EMAIL`로 로그인하고 wallet을 확인한 다음 **Grant signing permission**을 선택합니다. 이 단계를 완료하기 전에는 agent가 transaction에 sign할 수 없습니다.
2. 위에 출력된 wallet address를 사용하여 wallet에 자금을 입금합니다.
   - **Base Sepolia:** go to https://faucet.circle.com → select *Base Sepolia* → paste the wallet address
   - **Solana Devnet:** go to https://faucet.circle.com → select *Solana Devnet* → paste the wallet address
3. **WalletHub URL이 반환되지 않았고 instrument status가 `ACTIVE`이면** wallet에 signing 권한이 이미 부여된 것입니다. Payment가 성공하려면 faucet을 통한 입금은 여전히 필요합니다.

입금을 마치면 아래 셀에서 Enter를 눌러 계속합니다.

In [ ]:
input("Press Enter after you've funded the wallet and granted signing permission in WalletHub...")

### 3e단계 — Wallet Balance 검증

WalletHub 및 testnet faucet을 통해 wallet에 입금한 후 payment를 시도하기 전에 USDC balance를
검증합니다. Balance가 0이면 faucet transaction이 아직 pending 상태일 수 있으므로
1분 정도 기다렸다가 이 셀을 다시 실행하세요.

이 셀에서만 Notebook이 검증을 위해 로컬에서 잠시 `ProcessPaymentRole`을 사용합니다.
목적은 `GetPaymentInstrumentBalance` 호출뿐입니다. 실제 payment는 동일한 role로 실행되는
배포된 Runtime container 내부에서 수행됩니다. 이 API는 Ethereum 및 Solana network의
Coinbase CDP 및 StripePrivy wallet을 모두 지원합니다.

In [ ]:
# GetPaymentInstrumentBalance 호출을 위해 여기서 잠시 ProcessPaymentRole 사용
# Runtime에 배포된 agent는 동일한 role의 권한을 자동 사용
print("Assuming ProcessPaymentRole for balance check...")
balance_check_session = assume_role(PROCESS_PAYMENT_ROLE_ARN, f"balance-check-{int(datetime.now().timestamp())}")
balance_check_client = balance_check_session.client("bedrock-agentcore", endpoint_url=DP_ENDPOINT)

try:
    balance_resp = balance_check_client.get_payment_instrument_balance(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=PAYMENT_CONNECTOR_ID,
        paymentInstrumentId=PAYMENT_INSTRUMENT_ID,
        userId=USER_ID,
        chain="BASE_SEPOLIA",
        token="USDC",
    )
    token_balance = balance_resp.get("tokenBalance", {})
    if token_balance:
        amount_units = int(token_balance.get("amount", 0))
        decimals = token_balance.get("decimals", 6)
        readable = amount_units / (10**decimals)
        print(
            f"✅ Wallet balance: {readable:.6f} {token_balance.get('token', 'USDC')} on {token_balance.get('chain', 'unknown')}"
        )
    else:
        print("⚠️  Balance returned empty — faucet may still be pending")
    print(f"   Instrument ID: {PAYMENT_INSTRUMENT_ID}")
except Exception as e:
    print(f"⚠️  GetPaymentInstrumentBalance failed: {e}")
    print("   Verify bedrock-agentcore:GetPaymentInstrumentBalance is in the ProcessPaymentRole policy.")
    print("   Continue to Step 4 if the wallet is funded.")

## 4단계 — Payment Session 생성

Payment session은 agent 지출에 payment limit과 시간 제한 authorization을 적용합니다.
Notebook은 `ManagementRole`을 통해 session을 생성하며 agent는
`ProcessPaymentRole`만 사용하므로 session을 생성하거나 수정할 수 없습니다.

Payment limit을 제어하려면 `.env`에서 `SESSION_MAX_SPEND`와 `SESSION_EXPIRY_MINUTES`를 설정합니다.

In [ ]:
import uuid

session_response = mgmt_client.create_payment_session(
    paymentManagerArn=MANAGER_ARN,
    # userId는 X-Amzn-Bedrock-AgentCore-Payments-User-Id HTTP header에 매핑
    userId=USER_ID,
    expiryTimeInMinutes=SESSION_EXPIRY_MINUTES,
    limits={
        "maxSpendAmount": {
            "value": SESSION_MAX_SPEND,  # string이어야 함
            "currency": "USD",  # USDC가 아닌 USD - service가 budget 적용 시 변환
        }
    },
    clientToken=str(uuid.uuid4()),
)

payment_session = session_response["paymentSession"]
SESSION_ID = payment_session["paymentSessionId"]

print("✅ Payment session created")
print(f"   Session ID:  {SESSION_ID}")
print(f"   Payment limit: ${SESSION_MAX_SPEND} USD")
print(f"   Expires:     {SESSION_EXPIRY_MINUTES} minutes from now")

if "availableLimits" in payment_session:
    available = payment_session["availableLimits"]["availableSpendAmount"]
    print(f"   Available:   {available['value']} {available['currency']}")

> **참고: 유효한 두 가지 x402 browser pattern**
>
> [AgentCore Browser 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-browser.html)는 raw Playwright로 HTTP 402 response를 intercept하는 browser pattern을 보여 줍니다(`page.on("response", ...)` → `PaymentClient.process_payment` → retry 시 auth header를 삽입하는 `page.route()`). 이 pattern은 browser를 통해 액세스하는 API endpoint에 적합합니다.
>
> 이 Notebook은 NYT, FT, Substack 등 실제 consumer paywall이 HTTP 402를 반환하지 않으므로 다른 pattern을 사용합니다. 이러한 site는 page를 정상적으로 rendering하고(HTTP 200), article preview를 표시하며 premium content를 payment UI 뒤에 숨깁니다. 이를 반영하여 여기의 content provider는 page DOM(rendering된 HTML 구조), 구체적으로 agent가 page load 후 읽는 `<script id="x402-requirement">` tag에 x402 requirement가 embedded된 HTTP 200을 반환합니다.
>
> Response에 HTTP 402가 없으므로 plugin의 auto-intercept hook이 실행되지 않습니다. 대신 agent가 page에서 requirement를 직접 읽고 `PaymentManager.generate_payment_header`를 호출하여 proof를 생성합니다. 내부 `ProcessPayment` API는 동일하며 requirement의 출처만 다릅니다.

## 4b단계 — Payment Manager Observability 활성화

[AgentCore Payments observability 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-observability.html)에 따르면 Payment Manager telemetry는 vended log delivery를 통해 **resource별 opt-in** 방식으로 활성화합니다. 활성화하면 Payment Manager가 session, transaction, API별 metric과 함께 **AgentCore Observability → Payments** dashboard에 표시됩니다.

문서에서 설명하는 4단계 설정(0~4단계)에 *Agents using Payments* counter를 채우는 데 필요한 agent 측 구성을 추가합니다.

> **Browser 주의 사항:** 현재 AgentCore Browser는 resource별 vended log delivery를 지원하지 않습니다(`PutDeliverySource`는 browser ARN에 대해 *valid resource types are runtime / gateway / memory / payment-manager / code-interpreter / workload-identity*를 반환하며 거부). Browser tool action은 OTEL distro를 통해 agent trace 내부에 `browser session start`, `navigate`, `cleanup` span으로 계속 표시되지만 별도의 Browser service dashboard는 현재 없습니다.

In [ ]:
# Payment Manager에 4단계 vended log delivery 설정 실행
# Idempotent하게 동작하며 resource가 이미 있는 단계는 생략
import boto3

logs = boto3.client("logs", region_name=REGION)

LG_NAME = f"/aws/vendedlogs/bedrock-agentcore/{MANAGER_ARN.split('/')[-1].split('-')[0]}"
LG_ARN = f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LG_NAME}"

# 0단계 — log group
try:
    logs.create_log_group(logGroupName=LG_NAME)
    logs.put_retention_policy(logGroupName=LG_NAME, retentionInDays=30)
    print(f"✅ Created log group {LG_NAME}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"  ↻ Log group exists: {LG_NAME}")

# 1단계 — application log delivery source
try:
    logs.put_delivery_source(
        name="payforcontent-payments-logs",
        resourceArn=MANAGER_ARN,
        logType="APPLICATION_LOGS",
    )
    print("✅ Logs delivery source created")
except logs.exceptions.ConflictException:
    print("  ↻ Logs delivery source exists")

# 2단계 — trace delivery source
try:
    logs.put_delivery_source(
        name="payforcontent-payments-traces",
        resourceArn=MANAGER_ARN,
        logType="TRACES",
    )
    print("✅ Traces delivery source created")
except logs.exceptions.ConflictException:
    print("  ↻ Traces delivery source exists")

# 3a단계 — CloudWatch Logs delivery destination
try:
    logs.put_delivery_destination(
        name="payforcontent-payments-logs-dest",
        deliveryDestinationType="CWL",
        deliveryDestinationConfiguration={"destinationResourceArn": LG_ARN},
    )
    print("✅ Logs delivery destination created")
except logs.exceptions.ConflictException:
    print("  ↻ Logs delivery destination exists")

# 3b단계 — X-Ray trace destination
try:
    logs.put_delivery_destination(
        name="payforcontent-payments-traces-dest",
        deliveryDestinationType="XRAY",
    )
    print("✅ Traces delivery destination created")
except logs.exceptions.ConflictException:
    print("  ↻ Traces delivery destination exists")

# 4a단계 — log source → destination 연결
try:
    logs.create_delivery(
        deliverySourceName="payforcontent-payments-logs",
        deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:payforcontent-payments-logs-dest",
    )
    print("✅ Logs delivery created")
except logs.exceptions.ConflictException:
    print("  ↻ Logs delivery exists")

# 4b단계 — trace source → destination 연결
try:
    logs.create_delivery(
        deliverySourceName="payforcontent-payments-traces",
        deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:payforcontent-payments-traces-dest",
    )
    print("✅ Traces delivery created")
except logs.exceptions.ConflictException:
    print("  ↻ Traces delivery exists")

print()
print("Payment Manager observability enabled. After your first invoke,")
print("the AgentCore Observability → Payments tab will populate with this manager.")

## 5단계 — AgentCore Runtime에 Agent 배포

Agent 코드는 [`agent/payment_agent.py`](agent/payment_agent.py)에 있습니다.
`BedrockAgentCoreApp`으로 래핑되고 `@app.entrypoint`로 decorate된 Strands agent입니다.
Invocation payload에서 payment session, instrument, manager ARN, user ID, target paywall URL을
받으며 agent 내부에는 `assume_role`이 없습니다.

**AgentCore CLI**(`agentcore`)를 사용하여 agent를 container로 package하고 CodeBuild를 통해
ECR에 push한 다음 `ProcessPaymentRole`을 execution role로 사용하는 `AgentRuntime`을
생성합니다(local Docker 불필요).

**Build 선택 — CodeZip이 아닌 Container:** 이 사용 사례에는 `AgentCoreBrowser`를 위한
Playwright가 필요하며, Playwright에 포함된 Node.js driver binary는 CodeZip artifact로
package할 때 executable bit가 사라집니다. Container build는 CodeBuild에서 실행되어
file mode를 보존하고 Dockerfile 단계에서 Playwright driver에 `chmod`를 다시 적용할 수 있습니다.

**Lifecycle 설정:** `idleRuntimeSessionTimeout=600`(10분) 및 `maxLifetime=1800`
(30분)을 설정하여 1~3분의 browser flow에 충분한 여유를 제공하면서 microVM을
필요 이상으로 오래 유지하지 않습니다.

**Python 3.13:** 현재 CLI 기본값은 PYTHON_3_14이지만 Strands + anyio 조합에서
`weakref.NoneType` bug가 발생하므로 PYTHON_3_13으로 고정합니다.

In [ ]:
import subprocess
import shutil
import json as _json

AGENT_NAME = os.environ.get("AGENT_NAME", "PayForContentBrowserAgent")
PROJECT_NAME = os.environ.get("AGENT_PROJECT_NAME", "payforcontent")
RUNTIME_DIR = PROJECT_NAME


def run(cmd, **kw):
    """명령을 실행하고 실패하면 stdout/stderr를 표시합니다."""
    result = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if result.returncode != 0:
        print("stdout:", result.stdout[-500:])
        print("stderr:", result.stderr[-500:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result


# ── 5a. Project scaffold 생성(멱등) ─────────────────────────────────────────
if not os.path.isdir(RUNTIME_DIR):
    print(f"Scaffolding {RUNTIME_DIR}/ ...")
    run(
        [
            "agentcore",
            "create",
            "--name",
            AGENT_NAME,
            "--project-name",
            PROJECT_NAME,
            "--defaults",
            "--no-agent",  # 아래에서 자체 BYO agent 추가
            "--skip-git",
            "--skip-python-setup",
            "--skip-install",
            "--json",
        ]
    )

    # 5a.i. 기존 코드를 가리키는 Container build로 agent 추가
    run(
        [
            "agentcore",
            "add",
            "agent",
            "--type",
            "byo",
            "--name",
            AGENT_NAME,
            "--build",
            "Container",
            "--language",
            "Python",
            "--framework",
            "Strands",
            "--model-provider",
            "Bedrock",
            "--code-location",
            f"app/{AGENT_NAME}",
            "--entrypoint",
            "main.py",
            "--network-mode",
            "PUBLIC",
            "--protocol",
            "HTTP",
            "--idle-timeout",
            "600",
            "--max-lifetime",
            "1800",
            "--json",
        ],
        cwd=RUNTIME_DIR,
    )

# ── 5b. Agent 코드, requirements, Dockerfile을 scaffold에 복사 ───────────────
agent_dst = os.path.join(RUNTIME_DIR, "app", AGENT_NAME)
os.makedirs(agent_dst, exist_ok=True)
shutil.copy("agent/payment_agent.py", os.path.join(agent_dst, "main.py"))
shutil.copy("agent/requirements.txt", os.path.join(agent_dst, "requirements.txt"))
shutil.copy("agent/Dockerfile", os.path.join(agent_dst, "Dockerfile"))
print(f"✅ Copied agent + Dockerfile into {agent_dst}/")

# ── 5c. Runtime config에서 executionRoleArn + Python 3.13 고정 ───────────────
config_path = os.path.join(RUNTIME_DIR, "agentcore", "agentcore.json")
with open(config_path) as f:
    project_config = _json.load(f)

found = False
for runtime in project_config.get("runtimes", []):
    if runtime.get("name") == AGENT_NAME:
        runtime["executionRoleArn"] = PROCESS_PAYMENT_ROLE_ARN
        runtime["runtimeVersion"] = "PYTHON_3_13"
        found = True
        break
if not found:
    raise RuntimeError(f"Could not find runtime '{AGENT_NAME}' in {config_path}")

with open(config_path, "w") as f:
    _json.dump(project_config, f, indent=2)
print(f"✅ executionRoleArn = {PROCESS_PAYMENT_ROLE_ARN}")
print("✅ runtimeVersion   = PYTHON_3_13")

# ── 5d. Deployment target 설정(account + region) ─────────────────────────────
targets_path = os.path.join(RUNTIME_DIR, "agentcore", "aws-targets.json")
with open(targets_path, "w") as f:
    _json.dump(
        [
            {
                "name": "default",
                "description": "Pay for Content (Browser Use) — Runtime deployment",
                "account": ACCOUNT_ID,
                "region": REGION,
            }
        ],
        f,
        indent=2,
    )
print(f"✅ Deployment target: {ACCOUNT_ID} / {REGION}")

# ── 5e. CLI가 배포 시 필요한 CDK npm dependency 설치 ────────────────────────
cdk_dir = os.path.join(RUNTIME_DIR, "agentcore", "cdk")
if not os.path.isdir(os.path.join(cdk_dir, "node_modules")):
    print(f"Installing CDK npm deps in {cdk_dir}/ ...")
    run(["npm", "install", "--silent"], cwd=cdk_dir)
    print("✅ CDK deps installed")

### 5f — AWS에 배포

`agentcore deploy`는 CDK stack을 synthesize하고 Docker image를 build하여 ECR에 push하는
CodeBuild를 시작한 다음 AgentRuntime을 생성합니다.

> **첫 배포에는 5~10분이 걸릴 수 있습니다**(대부분 CodeBuild image build 시간).

In [ ]:
print("Deploying to AgentCore Runtime — this can take 5-10 minutes (CodeBuild)...")
run(["agentcore", "deploy", "--yes"], cwd=RUNTIME_DIR)
print("✅ Agent deployed")

In [ ]:
# 배포된 agent runtime ARN 저장
# `agentcore status --json`은 resource 목록을 반환하며 여기서는 해당 agent 선택
# CLI가 JSON document 뒤에 JSON이 아닌 upgrade notice를 추가할 수 있으므로
# 첫 JSON 값을 찾아 decoder가 해당 값만 읽도록 함
status_proc = subprocess.run(
    ["agentcore", "status", "--type", "agent", "--json"],
    cwd=RUNTIME_DIR,
    capture_output=True,
    text=True,
    check=True,
)
stdout = status_proc.stdout
start = next((i for i, ch in enumerate(stdout) if ch in "{["), -1)
if start < 0:
    raise RuntimeError(f"Could not find JSON in agentcore status output:\n{stdout}")
status, _ = _json.JSONDecoder().raw_decode(stdout[start:])
entries = status if isinstance(status, list) else status.get("resources", [])

agent_runtime_arn = None
for entry in entries:
    name = entry.get("name") or entry.get("agentName")
    if name == AGENT_NAME:
        agent_runtime_arn = (
            entry.get("identifier") or entry.get("agentRuntimeArn") or entry.get("runtimeArn") or entry.get("arn")
        )
        break

if not agent_runtime_arn:
    print("Raw status output:")
    print(_json.dumps(status, indent=2))
    raise RuntimeError("Could not locate agent runtime ARN in status output")

AGENT_RUNTIME_ARN = agent_runtime_arn
print(f"✅ Agent Runtime ARN: {AGENT_RUNTIME_ARN}")

## 6단계 — 배포된 Agent 호출

Payload의 payment context와 함께 `InvokeAgentRuntime`을 호출합니다. Agent는 자체 microVM에서
실행되어 browser를 제어하고 `ProcessPaymentRole`로 payment를 처리한 후 잠금 해제된
content를 반환합니다.

Container의 ambient credentials는 `ProcessPaymentRole`이며 agent 코드는
`sts:AssumeRole`을 호출하지 않습니다. Role 분리는 코드가 아니라 Runtime infrastructure에서 적용됩니다.

In [ ]:
import json

invoke_payload = {
    "prompt": (
        f"Please retrieve the premium article from {PAYWALL_DEMO_URL}. "
        f"Pay for it using x402 and give me a summary of what it contains."
    ),
    "paywall_url": PAYWALL_DEMO_URL,
    "payment_manager_arn": MANAGER_ARN,
    "user_id": USER_ID,
    "payment_session_id": SESSION_ID,
    "payment_instrument_id": PAYMENT_INSTRUMENT_ID,
}

response = mgmt_client.invoke_agent_runtime(
    agentRuntimeArn=AGENT_RUNTIME_ARN,
    payload=json.dumps(invoke_payload).encode("utf-8"),
    contentType="application/json",
    accept="application/json",
)

result_bytes = (
    response["response"].read() if hasattr(response.get("response"), "read") else response.get("response", b"")
)
result = json.loads(result_bytes.decode("utf-8")) if result_bytes else {}
print(result.get("response", result))

### Payment 기록 검증

`ManagementRole`로 session을 확인하여 agent의 지출이 기록되었는지 검증합니다.

In [ ]:
session_check = mgmt_client.get_payment_session(
    paymentManagerArn=MANAGER_ARN,
    paymentSessionId=SESSION_ID,
    userId=USER_ID,
)

session_data = session_check["paymentSession"]
print("✅ Session verified")
print(f"   Session ID:  {session_data['paymentSessionId']}")
print(f"   Payment limit: ${SESSION_MAX_SPEND} USD")
if "availableLimits" in session_data:
    remaining = session_data["availableLimits"]["availableSpendAmount"]
    print(f"   Remaining:   {remaining['value']} {remaining['currency']}")

## 7단계 — Observability

AgentCore는 이 사용 사례의 모든 계층에서 structured telemetry를 전송합니다. 4b단계를 실행하면 아래 네 가지 signal이 모두 **AgentCore Observability** dashboard에 표시됩니다.

| 계층 | Source | 표시 위치 |
|---|---|---|
| Runtime | `agentcore deploy` enables OTEL via the container's `opentelemetry-instrument` CMD | Bedrock AgentCore → All traces |
| Agent(Strands) | Strands가 Runtime distro를 통해 OTEL span 전송 | 각 trace의 waterfall 내부 |
| Browser tool | Strands `AgentCoreBrowser` tool이 client-side span 전송(start session, navigate, cleanup) | 각 trace의 waterfall 내부(별도 Browser dashboard는 아직 없음) |
| Payment Manager | Vended log + X-Ray span(4b단계 설정) | Bedrock AgentCore → **Payments** tab |

### Agent attribution(`payment_agent_name`)

Dashboard의 *Agents using Payments* counter는 **각 Payments API 호출에 `X-Amzn-Bedrock-AgentCore-Payments-Agent-Name` HTTP header가 포함된 경우에만** 증가합니다. `agent_name=`과 함께 `PaymentManager` 및 `AgentCorePaymentsPluginConfig`를 생성하면 SDK가 이를 자동으로 삽입합니다. `agent/payment_agent.py`는 container environment에서 `AGENT_NAME`을 읽어 두 객체에 모두 전달하므로 모든 span에 `payment_agent_name=PayForContentBrowserAgent`가 포함되고 dashboard에서 activity를 이 agent에 attribution합니다.

### 확인 위치

1. `us-west-2`에서 [CloudWatch GenAI Observability console](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability:agent-core)을 엽니다.
2. **Payments tab** — Payment Manager table, session 수, API invocation 수, error rate, *Agents using Payments* attribution
3. **All traces tab** — 모든 agent의 trace ID 목록
4. Trace를 선택하여 전체 waterfall 확인: `execute_event_loop_cycle` → `chat` → `execute_tool browser` → `execute_tool process_x402_payment` → `ProcessPayment`

아래는 dashboard에 표시된 성공한 Pay for Content session의 예입니다.

<div style="text-align:left">
    <strong>Payments tab</strong><br/>
    <img src="images/observability_payment_dashboard.png" width="80%"/>
</div>

<div style="text-align:left">
    <strong>All traces</strong><br/>
    <img src="images/observability_session_trace.png" width="80%"/>
</div>

<div style="text-align:left">
    <strong>Trace detail (one invocation)</strong><br/>
    <img src="images/observability_trace_detail.png" width="80%"/>
</div>

> **참고:** Wallet address와 proof signature가 trace input/output에 표시됩니다.
> Screenshot을 공유할 때는 wallet address와 `PAYMENT-SIGNATURE` header 값을 가리세요.

Runtime은 `bedrock-agentcore` CloudWatch namespace에도 metric을 게시합니다.
Session 수, invocation duration, tool error rate를 기반으로 budget 초과 또는 지속적인
payment 실패를 감지하는 alarm을 구성할 수 있습니다.

## 리소스 정리

4단계에서 생성한 payment session에는 기본 60분의 expiry가 내장되어 있습니다.
만료된 후에는 agent가 해당 session을 통해 지출할 수 없습니다.

AgentCore Runtime 배포에는 compute, storage, ECR 요금이 발생합니다. 작업을 마치면
다음과 같이 제거합니다.

```bash
cd PayForContentRuntime && agentcore remove all -y
```

이 명령은 AgentRuntime, ECR repository, CodeBuild project, CLI에서 생성한
CloudWatch log group을 제거합니다.

Account를 완전히 정리하려면 AWS CLI 또는 boto3를 통해 payment manager, connector,
instrument, credential provider도 삭제할 수 있습니다.

In [ ]:
# Session의 남은 limit과 만료 전 시간 검증
# Session은 만료 후 자동으로 payment 수락을 중단하므로 API 호출 불필요
session_check = mgmt_client.get_payment_session(
    paymentManagerArn=MANAGER_ARN,
    paymentSessionId=SESSION_ID,
    userId=USER_ID,
)
session_data = session_check["paymentSession"]
print(f"Session ID:    {session_data['paymentSessionId']}")
if "availableLimits" in session_data:
    remaining = session_data["availableLimits"]["availableSpendAmount"]
    print(f"Remaining:     {remaining['value']} {remaining['currency']}")

## 공동 책임

| 책임 항목                      | AWS / AgentCore                                          | 사용자                                              |
|:------------------------------|:---------------------------------------------------------|:----------------------------------------------------|
| Runtime container 격리        | Session별 microVM, 자동 제거                             | Workload에 맞게 `idleTimeout`, `maxLifetime` 설정   |
| Payment signing key           | AgentCore identity에서 보관 / Coinbase CDP 위임          | CDP project에서 Delegated Signing 활성화            |
| 지출 limit                    | Service가 session별 `maxSpendAmount` 적용                | Task에 적합한 session별 budget 설정                 |
| IAM role 분리                 | Runtime이 지정된 execution role 사용                    | Least-privilege role policy 작성(`setup_roles.sh` 참조) |
| Observability ingestion       | Trace + metric 자동 전송                                 | 필요한 metric에 alarm 구성                          |
| Wallet 입금                   | AgentCore에서 embedded wallet provision                 | Faucet(testnet) 또는 onramp(production)를 통해 입금 |
| Browser session 보안          | Containerized Chromium, ephemeral, optional recording    | Agent를 통한 production account 로그인 방지        |

# 축하합니다!

AgentCore Runtime에 managed hosting, role 분리 execution, built-in observability를 갖춘
end-to-end agentic payment 사용 사례를 배포했습니다. Agent는 자동으로 paywall page를
탐색하고 x402를 통해 결제한 후 잠금 해제된 content를 반환했으며 지출은 사용자가 생성한
session으로 제한되었습니다.